# Projet SM604 - De la classification des chiffres manuscrits à la détection de cancers du sein
## Mathématiques pour le Machine Learning - EFREI Paris 2025-2026
### Sabrina El Hassani, Aude Labat, Thomas Duriaud, Paul Fontaine, Evan Ladeira

---

## Partie 3 - Application au diagnostic médical (CBIS-DDSM)

### L'objectif de cette partie : comprendre un échec avant de le corriger

Cette partie a une vocation **analytique**. Plutôt que de présenter un modèle qui marche par chance,
nous documentons un phénomène que nous avons observé de façon reproductible, puis nous en expliquons
la cause profonde et les pistes de correction.

**Le phénomène :** selon la pondération de la fonction de coût, notre classifieur bascule vers l'une
ou l'autre de deux solutions dégénérées :
- soit il prédit **tout bénin** (et obtient une sensibilité de 0%),
- soit il prédit **tout malin** (et obtient une spécificité de 0%).

Dans les deux cas, le modèle **ne discrimine pas** : il choisit une classe par défaut. Nous allons
**reproduire volontairement ces deux cas**, montrer leurs matrices de confusion, puis expliquer
pourquoi cela arrive, quelles sont les limites du dataset, et comment un prétraitement comme CLAHE
permettrait d'y remédier.

### Rappel : l'asymétrie médicale

| Erreur | Prédiction | Réalité | Conséquence |
|--------|-----------|---------|-------------|
| **Faux négatif (FN)** | Bénin | Malin | **Cancer manqué → pronostic vital engagé** |
| **Faux positif (FP)** | Malin | Bénin | Fausse alarme → biopsie inutile, sans danger vital |

L'objectif clinique est de **maximiser la sensibilité** (minimiser les FN), puis de limiter les FP.

### Plan

**Section 3.1** Chargement des ROI crops et constat du déséquilibre  
**Section 3.2** Le modèle pondéré et les deux cas dégénérés (tout malin / tout bénin)  
**Section 3.3** Analyse : pourquoi cela arrive et limites du dataset  
**Section 3.4** Pistes d'amélioration (dont CLAHE) et ce qu'elles apporteraient

---
## Section 3.1 - Chargement et constat du déséquilibre

### Imports

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.metrics import confusion_matrix

np.random.seed(42)
print("Imports OK")

### Chemins du dataset

**Deux chemins à adapter :**
- `CSV_DIR`  : dossier contenant les fichiers `.csv`
- `JPEG_DIR` : dossier contenant les sous-dossiers `1.3.6.1.4.1.9590.xxx/`

In [ ]:
# --- Adapter ces deux chemins ---
CSV_DIR  = "C:/Users/audel/Documents/csv"    # dossier contenant les .csv
JPEG_DIR = "C:/Users/audel/Documents/jpeg"   # dossier contenant les sous-dossiers 1.3.6...
IMG_SIZE = 128   # 128x128 suffit pour cette demonstration (plus rapide a charger)

for d in [CSV_DIR, JPEG_DIR]:
    print(f"  {d} : {'OK' if os.path.isdir(d) else 'INTROUVABLE'}")

### Chargement des ROI crops

On charge les `cropped image file path` (ROI autour de la lésion) via la jointure sur
`dicom_info.csv` (filtré sur les crops). Le 3ème segment du chemin DICOM est le `SeriesInstanceUID`
qui sert de clé de jointure.

In [ ]:
def charger_cbis_roi(csv_masse, csv_dicom, jpeg_dir, taille=128):
    """
    Charge les ROI crops du dataset CBIS-DDSM.

    Etapes :
    1. Lire le CSV masse, binariser (BENIGN/BWC -> 0, MALIGNANT -> 1)
    2. Extraire le SeriesInstanceUID (3e segment du chemin DICOM du crop)
    3. Joindre avec dicom_info.csv (SeriesDescription='cropped images')
    4. Charger chaque image en niveaux de gris, redimensionner, normaliser [0,1]

    Retourne : X array (n, taille, taille) float32, y array (n,) int
    """
    df_masse = pd.read_csv(csv_masse)
    df_dicom = pd.read_csv(csv_dicom)

    def label(p):
        p = str(p).strip().upper()
        if p in ["BENIGN", "BENIGN_WITHOUT_CALLBACK"]: return 0
        if p == "MALIGNANT": return 1
        return -1

    df_masse["label"] = df_masse["pathology"].apply(label)
    df_masse = df_masse[df_masse["label"] != -1].reset_index(drop=True)

    df_masse["UID_crop"] = df_masse["cropped image file path"].apply(
        lambda p: p.replace("\\", "/").split("/")[2]
    )
    df_crops = df_dicom[
        df_dicom["SeriesDescription"].str.contains("cropped", case=False, na=False)
    ][["SeriesInstanceUID", "image_path"]].drop_duplicates("SeriesInstanceUID")
    df_masse = df_masse.merge(
        df_crops, left_on="UID_crop", right_on="SeriesInstanceUID", how="left"
    )
    df_masse = df_masse[df_masse["image_path"].notna()].reset_index(drop=True)
    df_masse["chemin_jpeg"] = df_masse["image_path"].apply(
        lambda p: os.path.join(jpeg_dir, "/".join(str(p).replace("\\", "/").split("/")[-2:]))
    )

    print(f"  {len(df_masse)} ROI crops | "
          f"Benin={int((df_masse['label']==0).sum())}  "
          f"Malin={int((df_masse['label']==1).sum())}")

    images, labels, n_manq = [], [], 0
    for i, row in df_masse.iterrows():
        try:
            img = Image.open(row["chemin_jpeg"]).convert("L")
            img = img.resize((taille, taille), Image.BICUBIC)
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(row["label"])
        except Exception:
            n_manq += 1
        if (i+1) % 100 == 0:
            print(f"    {len(images)} chargees...", end="\r")
    if n_manq > 0:
        print(f"    {n_manq} fichiers introuvables")
    return np.array(images, dtype=np.float32), np.array(labels, dtype=int)


print("Chargement du train...")
X_train, y_train = charger_cbis_roi(
    os.path.join(CSV_DIR, "mass_case_description_train_set.csv"),
    os.path.join(CSV_DIR, "dicom_info.csv"), JPEG_DIR, taille=IMG_SIZE)
print("\nChargement du test...")
X_test, y_test = charger_cbis_roi(
    os.path.join(CSV_DIR, "mass_case_description_test_set.csv"),
    os.path.join(CSV_DIR, "dicom_info.csv"), JPEG_DIR, taille=IMG_SIZE)
print(f"\nTrain : {X_train.shape}  |  Test : {X_test.shape}")

### Le déséquilibre, point de départ de notre analyse

On visualise la distribution des classes. C'est le premier ingrédient du phénomène que nous allons
observer : si une classe domine dans le jeu de test, prédire toujours cette classe donne déjà un
bon score apparent.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, y, titre in [(ax1, y_train, "Train"), (ax2, y_test, "Test")]:
    n_b, n_m = int((y==0).sum()), int((y==1).sum())
    barres = ax.bar(["Benin", "Malin"], [n_b, n_m], color=["#27ae60", "#c0392b"])
    ax.set_title(f"{titre} (n={len(y)})"); ax.set_ylabel("Nombre d'images")
    for barre, val in zip(barres, [n_b, n_m]):
        ax.text(barre.get_x()+barre.get_width()/2, val+3,
                f"{val}\n({100*val/len(y):.1f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0, max(n_b,n_m)*1.25)
plt.suptitle("Distribution des classes - CBIS-DDSM ROI crops", fontsize=13)
plt.tight_layout()
plt.savefig("distribution_classes.png", dpi=100, bbox_inches="tight")
plt.show()

pct_benin_test = 100*(y_test==0).mean()
print(f"Dans le TEST, {pct_benin_test:.1f}% des cas sont benins.")
print(f"=> un modele qui predit TOUT BENIN obtient deja {pct_benin_test:.1f}% de bonnes reponses,")
print(f"   mais une sensibilite de 0% (aucun cancer detecte). C'est le piege.")

### Préparation pour les modèles denses

In [ ]:
N_PIXELS = IMG_SIZE * IMG_SIZE

def one_hot(y, C=2):
    """Convertit etiquettes entieres en one-hot. y=0->[1,0], y=1->[0,1]."""
    n = len(y); Y = np.zeros((n, C)); Y[np.arange(n), y] = 1.0
    return Y

X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test),  -1)
Y_train_oh   = one_hot(y_train, C=2)

print(f"X_train_flat : {X_train_flat.shape}")
print(f"X_test_flat  : {X_test_flat.shape}")

---
## Section 3.2 - Le modèle pondéré et les deux cas dégénérés

### La fonction de coût pondérée

On reprend la cross-entropy pondérée par classe :

$$L_{\text{pond}} = -\frac{1}{n}\sum_{i=1}^{n} w_{y_i} \, \ln(P_{y_i}(\vec{x}_i))$$

où $w_{y_i}$ est le poids de la vraie classe de l'exemple $i$. Le gradient devient
$\delta_i = w_{y_i}(P_i - Y_i)$. Le paramètre clé est le rapport $w_{\text{malin}} / w_{\text{benin}}$ :

- s'il est **faible** (< 1) → le modèle privilégie la classe bénigne → **tout bénin**
- s'il est **élevé** (> quelques unités) → le modèle privilégie la classe maligne → **tout malin**

Nous allons démontrer ces deux comportements en faisant varier ce rapport.

### Fonctions communes

In [ ]:
def softmax_python(o):
    """Softmax stable. o (n,C) -> P (n,C)."""
    o = np.array(o); res = []
    for image in o:
        m = max(image)
        ex = [math.exp(s - m) for s in image]; sm = sum(ex)
        res.append([e/sm for e in ex])
    return np.array(res)


def sensibilite(pred, y_true):
    """VP/(VP+FN) = proportion de cancers detectes."""
    mask = (y_true == 1)
    vp = int((pred[mask]==1).sum()); fn = int((pred[mask]==0).sum())
    return vp / (vp + fn + 1e-8)


def specificite(pred, y_true):
    """VN/(VN+FP) = proportion de benins correctement identifies."""
    mask = (y_true == 0)
    vn = int((pred[mask]==0).sum()); fp = int((pred[mask]==1).sum())
    return vn / (vn + fp + 1e-8)


def train_linear_pondere(X_train, Y_train, y_train, n_input, poids_classe,
                         n_classes=2, lr=0.01, batch_size=32, max_epochs=40):
    """
    Modele lineaire avec weighted cross-entropy.
    poids_classe : [w_benin, w_malin]. Le gradient est delta_i = w[y_i]*(P_i - Y_i).
    Retourne A, b.
    """
    A = np.random.randn(n_classes, n_input) * 0.01
    b = np.zeros(n_classes)
    n = len(X_train)
    for epoch in range(max_epochs):
        idx = np.random.permutation(n)
        X_s, Y_s, y_s = X_train[idx], Y_train[idx], y_train[idx]
        for start in range(0, n, batch_size):
            X_b = X_s[start:start+batch_size]
            Y_b = Y_s[start:start+batch_size]
            y_b = y_s[start:start+batch_size]
            P_b   = softmax_python(X_b @ A.T + b)
            w_ex  = poids_classe[y_b].reshape(-1, 1)   # poids par exemple
            delta = w_ex * (P_b - Y_b)                  # gradient pondere
            A -= lr * (delta.T @ X_b) / len(X_b)
            b -= lr * delta.mean(axis=0)
    return A, b


def predire(X_flat, A, b):
    """Predictions 0/1 du modele lineaire."""
    return np.argmax(softmax_python(X_flat @ A.T + b), axis=1)


print("Fonctions chargees.")

### CAS 1 — Forte pénalité sur les malins : le modèle prédit TOUT MALIN

On met une pénalité élevée sur les erreurs malignes ($w_{\text{malin}} = 5$). Le modèle, ne sachant
pas bien discriminer, trouve qu'il est moins coûteux de tout classer malin : il ne rate alors aucun
cancer (sensibilité 100%) mais déclenche une fausse alarme sur tous les bénins (spécificité 0%).

In [ ]:
np.random.seed(42)
poids_cas1 = np.array([1.0, 5.0])   # forte penalite sur les malins
A1, b1 = train_linear_pondere(X_train_flat, Y_train_oh, y_train,
                               n_input=N_PIXELS, poids_classe=poids_cas1)
pred_cas1 = predire(X_test_flat, A1, b1)

print(f"CAS 1 : poids = {poids_cas1.tolist()} (forte penalite malin)")
print(f"  Predits benins : {int((pred_cas1==0).sum())}  |  Predits malins : {int((pred_cas1==1).sum())}")
print(f"  Sensibilite : {sensibilite(pred_cas1, y_test)*100:.1f}%")
print(f"  Specificite : {specificite(pred_cas1, y_test)*100:.1f}%")

### CAS 2 — Pénalité standard : le modèle prédit TOUT BÉNIN

Avec des poids égaux ($w_{\text{malin}} = w_{\text{benin}} = 1$, soit la cross-entropy classique),
le modèle minimise l'erreur le plus facilement en pariant sur la classe majoritaire : il prédit
tout bénin. Spécificité 100% mais sensibilité 0% : aucun cancer détecté.

In [ ]:
np.random.seed(42)
poids_cas2 = np.array([1.0, 1.0])   # poids egaux = cross-entropy standard
A2, b2 = train_linear_pondere(X_train_flat, Y_train_oh, y_train,
                               n_input=N_PIXELS, poids_classe=poids_cas2)
pred_cas2 = predire(X_test_flat, A2, b2)

print(f"CAS 2 : poids = {poids_cas2.tolist()} (penalite standard)")
print(f"  Predits benins : {int((pred_cas2==0).sum())}  |  Predits malins : {int((pred_cas2==1).sum())}")
print(f"  Sensibilite : {sensibilite(pred_cas2, y_test)*100:.1f}%")
print(f"  Specificite : {specificite(pred_cas2, y_test)*100:.1f}%")

### Les deux cas côte à côte

Les matrices de confusion rendent le phénomène visuel : dans les deux cas, une **colonne entière**
est vide, signe que le modèle ne prédit qu'une seule classe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (titre, pred) in zip(axes, [("CAS 1 : tout malin (w_malin=5)", pred_cas1),
                                     ("CAS 2 : tout benin (w egaux)", pred_cas2)]):
    cm = confusion_matrix(y_test, pred)
    VN, FP, FN, VP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    sens, spec = VP/(VP+FN+1e-8), VN/(VN+FP+1e-8)
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["P.Benin","P.Malin"]); ax.set_yticklabels(["R.Benin","R.Malin"])
    ax.set_title(f"{titre}\nSensib={sens*100:.0f}%  Spec={spec*100:.0f}%", fontsize=11)
    for i,j,e,c in [(0,0,f"VN\n{VN}","#000"),(0,1,f"FP\n{FP}","#c0392b"),
                     (1,0,f"FN\n{FN}","#c0392b"),(1,1,f"VP\n{VP}","#000")]:
        ax.text(j,i,e,ha="center",va="center",fontsize=12,fontweight="bold",color=c)
plt.suptitle("Les deux solutions degenerees selon la ponderation", fontsize=13)
plt.tight_layout()
plt.savefig("deux_cas_degeneres.png", dpi=100, bbox_inches="tight")
plt.show()

### Balayage de la pénalité : le basculement d'une classe à l'autre

Pour montrer qu'il n'existe **aucun réglage intermédiaire satisfaisant**, on balaie la pénalité.
On observe que le modèle saute de « tout bénin » à « tout malin » sans jamais s'arrêter sur un
vrai compromis : c'est la signature d'un modèle qui ne discrimine pas.

In [ ]:
penalites = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]
sens_list, spec_list = [], []

print(f"{'w_malin':>8} {'Sensib':>9} {'Specif':>9}  comportement")
for pen in penalites:
    np.random.seed(42)
    A_t, b_t = train_linear_pondere(X_train_flat, Y_train_oh, y_train,
                                     n_input=N_PIXELS, poids_classe=np.array([1.0, pen]),
                                     max_epochs=30)
    pred = predire(X_test_flat, A_t, b_t)
    s, sp = sensibilite(pred, y_test), specificite(pred, y_test)
    sens_list.append(s*100); spec_list.append(sp*100)
    comport = "tout benin" if s < 0.1 else ("tout malin" if sp < 0.1 else "mixte")
    print(f"{pen:>8.1f} {s*100:>8.1f}% {sp*100:>8.1f}%  {comport}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(penalites, sens_list, "o-", color="darkgreen", label="Sensibilite")
ax.plot(penalites, spec_list, "s-", color="steelblue", label="Specificite")
ax.set_xlabel("Penalite sur les erreurs malignes (w_malin)")
ax.set_ylabel("%"); ax.set_ylim(-5, 105)
ax.set_title("Basculement brutal entre les deux classes degenerees")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("balayage_penalite.png", dpi=100, bbox_inches="tight")
plt.show()

print("Lecture : la sensibilite et la specificite sont quasi-complementaires (l'une a 100%,")
print("l'autre a 0%). Le modele bascule de 'tout benin' a 'tout malin' sans compromis stable.")
print("Un modele qui DISCRIMINE montrerait les deux courbes elevees simultanement.")

---
## Section 3.3 - Analyse : pourquoi cela arrive

### La cause profonde n'est pas la pondération

On pourrait croire qu'il suffit de trouver le bon poids. Le balayage ci-dessus montre que **non** :
il n'existe pas de réglage qui donne à la fois une bonne sensibilité ET une bonne spécificité. Le
modèle ne fait que choisir quelle classe prédire systématiquement.

**La vraie cause :** le modèle linéaire **n'arrive pas à séparer les deux classes** dans l'espace
des pixels bruts. Quand un classifieur ne peut pas tracer de frontière discriminante, sa stratégie
optimale pour minimiser la loss est de tout mettre dans une seule classe — et la pondération décide
simplement laquelle. C'est un symptôme, pas la maladie.

### Pourquoi le modèle ne peut pas discriminer : les limites du dataset et des images

**1. Le contraste local des mammographies est très faible.** Les textures qui distinguent une masse
bénigne d'une masse maligne (spiculations, irrégularité des contours, densité) se jouent sur des
variations de niveaux de gris très subtiles. Sur les pixels bruts, ces différences sont noyées : le
modèle « voit » surtout une tache grise floue dans les deux cas.

**2. Le redimensionnement détruit l'information fine.** Réduire un crop à 128×128 lisse les détails
diagnostiques. Les microcalcifications et les fines spiculations, qui font quelques pixels, peuvent
disparaître.

**3. Le volume de données est faible.** Avec ~1300 images d'entraînement (contre 60 000 pour MNIST,
50 000 pour CIFAR-10), un modèle a très peu d'exemples pour apprendre des frontières complexes,
surtout sur une tâche aussi subtile.

**4. La tâche est intrinsèquement difficile.** Même les radiologues experts ont un taux de désaccord
de 20-30% sur les cas ambigus. Distinguer bénin/malin sur une simple imagette n'est pas trivial,
même pour un humain entraîné.

**5. Le modèle linéaire est trop simple.** Il ne peut tracer qu'une frontière linéaire dans l'espace
des 16 384 pixels, alors que la frontière réelle bénin/malin est hautement non-linéaire.

In [ ]:
# Illustration : superposition des intensites moyennes des deux classes
# Si les deux distributions se chevauchent fortement, le modele ne peut pas separer.
moy_benin = X_train[y_train==0].mean(axis=0)   # image moyenne des benins
moy_malin = X_train[y_train==1].mean(axis=0)   # image moyenne des malins

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(moy_benin, cmap="gray"); axes[0].set_title("Image moyenne - Benins"); axes[0].axis("off")
axes[1].imshow(moy_malin, cmap="gray"); axes[1].set_title("Image moyenne - Malins"); axes[1].axis("off")
diff = np.abs(moy_benin - moy_malin)
im = axes[2].imshow(diff, cmap="hot"); axes[2].set_title("Difference absolue"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle("Les images moyennes des deux classes sont quasi-identiques", fontsize=13)
plt.tight_layout()
plt.savefig("images_moyennes.png", dpi=100, bbox_inches="tight")
plt.show()

print(f"Difference moyenne entre les deux classes : {diff.mean():.4f} (sur une echelle [0,1])")
print("Une difference aussi faible confirme que le signal discriminant est tres tenu")
print("dans les pixels bruts : c'est pourquoi le modele ne peut pas separer les classes.")

---
## Section 3.4 - Pistes d'amélioration

Maintenant que la cause est claire (le signal discriminant n'est pas exploitable dans les pixels
bruts), les pistes d'amélioration découlent logiquement. On les classe par ordre d'impact attendu.

### Piste 1 — CLAHE : rendre le signal visible (la plus importante)

**CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) est une égalisation d'histogramme
**locale** : elle amplifie le contraste région par région au lieu de globalement. Sur une
mammographie, cela fait ressortir les textures fines (spiculations, contours de masse) qui étaient
invisibles dans l'image brute.

**Ce que CLAHE nous permettrait de faire :** donner au modèle des images où la différence entre
bénin et malin est réellement visible. Avec un signal discriminant exploitable, le modèle pourrait
enfin tracer une vraie frontière entre les classes au lieu de tout prédire dans une seule. La
pondération retrouverait alors son rôle normal : ajuster finement le compromis sensibilité/
spécificité, et non choisir une classe par défaut. C'est le prétraitement utilisé par la quasi-
totalité des pipelines performants sur CBIS-DDSM.

Ci-dessous, une démonstration de l'effet de CLAHE sur une de nos images (implémentation maison,
documentée) pour visualiser concrètement le gain de contraste.

In [ ]:
def clahe_maison(img, n_tiles=8, clip_limit=2.0):
    """
    CLAHE implemente from scratch (demonstration de l'algorithme).

    Principe :
    1. Decouper l'image en n_tiles x n_tiles tuiles
    2. Pour chaque tuile : histogramme -> clipping (limite les pics) ->
       redistribution de l'exces -> CDF -> table de correspondance
    3. Appliquer la table de chaque tuile a ses pixels

    img        : array (H,W) float [0,1]
    n_tiles    : nombre de tuiles par dimension
    clip_limit : limite d'amplification du contraste
    Retourne : array (H,W) float [0,1]
    """
    H, W   = img.shape
    u8     = (img*255).astype(np.uint8)
    th, tw = H//n_tiles, W//n_tiles
    out    = np.zeros_like(u8, dtype=np.float32)
    for ti in range(n_tiles):
        for tj in range(n_tiles):
            y0, y1 = ti*th, ((ti+1)*th if ti<n_tiles-1 else H)
            x0, x1 = tj*tw, ((tj+1)*tw if tj<n_tiles-1 else W)
            tile = u8[y0:y1, x0:x1]
            hist = np.bincount(tile.flatten(), minlength=256).astype(np.float32)
            clip = clip_limit * tile.size / 256          # seuil de clipping
            exces = np.maximum(hist-clip, 0).sum()        # exces a redistribuer
            hist = np.minimum(hist, clip) + exces/256
            cdf  = hist.cumsum()
            cdf  = (cdf - cdf.min())/(cdf.max()-cdf.min()+1e-8)*255
            out[y0:y1, x0:x1] = cdf[tile]
    return out/255.0


# Demonstration sur une image maligne (la ou le contraste compte le plus)
idx_demo = np.where(y_train==1)[0][0]
img_brute = X_train[idx_demo]
img_clahe = clahe_maison(img_brute)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_brute, cmap="gray"); axes[0].set_title("Image brute (signal noye)"); axes[0].axis("off")
axes[1].imshow(img_clahe, cmap="gray"); axes[1].set_title("Apres CLAHE (textures revelees)"); axes[1].axis("off")
plt.suptitle("Ce que CLAHE apporterait : un contraste local exploitable", fontsize=13)
plt.tight_layout()
plt.savefig("demo_clahe.png", dpi=100, bbox_inches="tight")
plt.show()

### Piste 2 — Architecture convolutive (CNN)

Le modèle linéaire ne peut tracer qu'une frontière linéaire. Un **CNN** détecte des motifs locaux
(contours, textures) grâce à ses filtres de convolution, et les combine de façon non-linéaire. Couplé
à CLAHE, il pourrait exploiter les textures révélées pour réellement distinguer les classes. C'est
l'architecture qu'on a construite en Partie 2 et qui serait ici l'étape naturelle suivante.

### Piste 3 — Augmenter le volume de données

**Data augmentation** (retournements, rotations légères) : multiplie artificiellement le dataset
sans biaiser la distribution, ce qui réduit l'overfitting. Médicalement valide car une mammographie
reste plausible après un retournement horizontal (symétrie gauche/droite).

### Piste 4 — Transfer learning

Un réseau pré-entraîné sur ImageNet (ResNet, EfficientNet) apporte des détecteurs de formes et de
textures déjà appris sur des millions d'images. C'est la piste la plus puissante face au faible
volume de données médicales : elle atteint typiquement des AUC de 0.80-0.88 sur CBIS-DDSM, là où
nos modèles from scratch plafonnent.

### Piste 5 — Affiner le seuil de décision

Plutôt que de prédire la classe la plus probable (argmax = seuil 0.5), on pourrait régler le seuil
sur la probabilité de malignité pour calibrer précisément le point de fonctionnement
sensibilité/spécificité selon les exigences cliniques (courbe ROC).

---

### Conclusion

Le comportement « tout bénin / tout malin » n'est pas un bug de notre code ni un mauvais réglage de
pondération : c'est le **symptôme attendu** d'un modèle qui ne peut pas discriminer faute de signal
exploitable dans les pixels bruts. La pondération ne fait que choisir vers quelle classe dégénérée
le modèle bascule. La correction ne passe donc pas par la fonction de coût mais par le **prétraitement
des images** (CLAHE en tête) et par des **architectures capables d'exploiter la structure spatiale**
(CNN, transfer learning). Cette analyse illustre une leçon générale du machine learning appliqué :
**la qualité de la représentation des données prime souvent sur la sophistication du modèle**.